In [1]:
from huggingface_hub import login

# Run `huggingface-cli login` once in your terminal beforehand -- it caches the token
# permanently to disk, so login() below picks it up with no arguments needed.
try:
    login()
    print("✓ Hugging Face authentication successful!")
except Exception as e:
    print(f"⚠ Not logged in yet: {e}")
    print("Run this once in your terminal (outside the notebook): huggingface-cli login")

✓ Hugging Face authentication successful!


c:\Users\Ipsita\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset

ds = load_dataset("hassanjbara/BASEPROD", split="train", streaming=True)
print(ds)

IterableDataset({
    features: ['color', 'depth', 'depth_16bit', 'thermal', 'thermal_rgb'],
    num_shards: 106
})


In [3]:
#Print the available features by inspecting the schema of the dataset so preprocessing adapts to whatever columns actually exist.
print("Available features:")
for name, feature in ds.features.items():
    print(f"  {name}: {feature}")

Available features:
  color: Image(mode=None, decode=True)
  depth: Image(mode=None, decode=True)
  depth_16bit: Image(mode=None, decode=True)
  thermal: Image(mode=None, decode=True)
  thermal_rgb: Image(mode=None, decode=True)


In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm  # plain tqdm, not tqdm.notebook -- avoids the IProgress/ipywidgets requirement


# --- Preprocessing ---

def preprocess_color(image, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Resize an RGB color image (in its native mode) and scale to [0, 1]."""
    resized = image.convert("RGB").resize(size, Image.BILINEAR)
    return np.array(resized).astype(np.float32) / 255.0


def preprocess_depth(image, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Resize a depth map (in its native mode) and normalize to [0, 1].

    Resizing happens on the original PIL image BEFORE casting to float --
    Pillow cannot resize a 3-channel float32 array directly. If the depth
    image has multiple channels (e.g. depth encoded/visualized across RGB),
    it's collapsed to a single channel by averaging.
    """
    resized = image.resize(size, Image.NEAREST)
    arr = np.array(resized).astype(np.float32)
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    depth_range = arr.max() - arr.min()
    if depth_range == 0:
        return np.zeros_like(arr)
    return (arr - arr.min()) / depth_range


def preprocess_thermal(image, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Resize a thermal image (in its native mode) and scale to [0, 1].

    Same reasoning as preprocess_depth: resize first, cast to float after.
    """
    resized = image.resize(size, Image.BILINEAR)
    arr = np.array(resized).astype(np.float32)
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    therm_range = arr.max() - arr.min()
    if therm_range == 0:
        return np.zeros_like(arr)
    return (arr - arr.min()) / therm_range


# --- Feature extraction ---

def extract_color_features(color_arr: np.ndarray) -> dict:
    """Basic color statistics: per-channel mean/std plus a coarse histogram."""
    features = {}
    for i, channel in enumerate(["r", "g", "b"]):
        features[f"color_{channel}_mean"] = float(color_arr[..., i].mean())
        features[f"color_{channel}_std"] = float(color_arr[..., i].std())
    gray = color_arr.mean(axis=-1)
    hist, _ = np.histogram(gray, bins=8, range=(0.0, 1.0))
    for i, count in enumerate(hist):
        features[f"color_hist_bin_{i}"] = int(count)
    return features


def extract_depth_features(depth_arr: np.ndarray) -> dict:
    """Depth statistics: central tendency, spread, and coverage of valid pixels."""
    valid = depth_arr[depth_arr > 0]
    return {
        "depth_mean": float(valid.mean()) if valid.size else 0.0,
        "depth_std": float(valid.std()) if valid.size else 0.0,
        "depth_valid_ratio": float(valid.size / depth_arr.size),
    }


def extract_thermal_features(thermal_arr: np.ndarray) -> dict:
    """Thermal statistics: overall intensity spread as a proxy for hotspot presence."""
    return {
        "thermal_mean": float(thermal_arr.mean()),
        "thermal_std": float(thermal_arr.std()),
        "thermal_max": float(thermal_arr.max()),
    }


def extract_features(sample: dict) -> dict:
    """Run the full preprocessing + feature extraction pipeline on one sample."""
    features = {}
    if sample.get("color") is not None:
        features.update(extract_color_features(preprocess_color(sample["color"])))
    if sample.get("depth") is not None:
        features.update(extract_depth_features(preprocess_depth(sample["depth"])))
    if sample.get("thermal") is not None:
        features.update(extract_thermal_features(preprocess_thermal(sample["thermal"])))
    return features


# --- Run the pipeline ---

N_SAMPLES = 50  # adjust as needed; the dataset is streamed, so this pulls N_SAMPLES on demand

rows = [extract_features(sample) for sample in tqdm(ds.take(N_SAMPLES), total=N_SAMPLES)]
features_df = pd.DataFrame(rows)
print(f"Extracted features for {len(features_df)} samples")
features_df.head()

  0%|          | 0/50 [00:00<?, ?it/s]'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/hassanjbara/BASEPROD/resolve/7a6a0d0a4e0ce95d3b5bbbaba3a066c4f18d67e3/data/train-00000-of-00106.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/hassanjbara/BASEPROD/resolve/7a6a0d0a4e0ce95d3b5bbbaba3a066c4f18d67e3/data/train-00000-of-00106.parquet
Retrying in 2s [Retry 2/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/hassanjbara/BASEPROD/resolve/7a6a0d0a4e0ce95d3b5bbbaba3a066c4f18d67e3/data/train-00000-of-00106.parquet
Retrying in 4s [Retry 3/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/hassanjbara/BASEPROD/resolve/7a6a0d0a4e0ce95d3b5bbbaba3a066c4f18d67e3/data/train-00000-of-00106.parquet
Retrying in 8s [Retry 4/5].
'The read operation timed out' thrown while requesting GET https://hug

In [ ]:
output_path = "baseprod_features.csv"
features_df.to_csv(output_path, index=False)
print(f"Saved {len(features_df)} rows to {output_path}")

Saved 50 rows to baseprod_features.csv
